# HAI Feature Engineering   

Transforms the curated warehouse dataset into a flat feature matrix ready for scikit-learn classifiers. Follows a strict order to prevent data leakage:  
1. Split
2. Engineer Features
3. Normalize

### This notebook should only be ran after HAI Dataset Pipeline notebook

## 0. Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# ── Paths ────────────────────────────────────────────────────────────────────
WORK_DIR    = Path("work")
PROJECT_DIR = WORK_DIR / "hai_21_03"
DATA_DIR    = PROJECT_DIR / "data"
WH_DIR      = DATA_DIR / "warehouse"
FE_DIR      = DATA_DIR / "features"
FE_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")

# ── Load curated dataset ─────────────────────────────────────────────────────
curated_files = sorted(WH_DIR.glob("hai_curated_*.parquet"))
if not curated_files:
    raise FileNotFoundError(f"No curated parquet found in {WH_DIR}")

curated_path = curated_files[-1]
print(f"Loading: {curated_path}")
df = pd.read_parquet(curated_path)
print(f"Shape:   {df.shape}")
print(f"Labels:  {df['label'].value_counts().sort_index().to_dict()}")

Loading: work/hai_21_03/data/warehouse/hai_curated_20260221_135249_utc.parquet
Shape:   (1323608, 91)
Labels:  {0: 1178988, 1: 8947, 2: 135673}


## 1. Time-Based Train / Validation / Test Split

Split by timestamp order: 
* 60% train
* 20% validation
* 20% test

Random splitting is explicitly avoided. It would allow future readings to inform past predictions, inflating performance estimates.  
All feature statistics and normalizations parameters will be computed on the train split only

In [3]:
print(f"\nLabel distribution by split:")
for name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    counts = split_df['label'].value_counts().sort_index()
    pcts   = (counts / len(split_df) * 100).round(2)
    for label, label_name in [(0, 'normal'), (1, 'attack'), (2, 'fault')]:
        n   = counts.get(label, 0)
        pct = pcts.get(label, 0.0)
        print(f"  {name:<6} {label_name}: {n:>8,}  ({pct:.2f}%)")
    print()
    


Label distribution by split:
  Train  normal:  729,244  (91.83%)
  Train  attack:    8,947  (1.13%)
  Train  fault:   55,973  (7.05%)

  Val    normal:  236,555  (89.36%)
  Val    attack:        0  (0.00%)
  Val    fault:   28,167  (10.64%)

  Test   normal:  213,189  (80.53%)
  Test   attack:        0  (0.00%)
  Test   fault:   51,533  (19.47%)



In [2]:
# Sort by timestamp to ensure correct temporal ordering
df = df.sort_values("timestamp").reset_index(drop=True)

# Compute split boundaries by row position
n = len(df)
train_end = int(n * 0.60)
val_end   = int(n * 0.80)

df_train = df.iloc[:train_end].copy()
df_val   = df.iloc[train_end:val_end].copy()
df_test  = df.iloc[val_end:].copy()

print(f"Total rows:      {n:,}")
print(f"Train:           {len(df_train):,}  ({len(df_train)/n*100:.1f}%)  "
      f"{df_train['timestamp'].min().date()} → {df_train['timestamp'].max().date()}")
print(f"Validation:      {len(df_val):,}  ({len(df_val)/n*100:.1f}%)  "
      f"{df_val['timestamp'].min().date()} → {df_val['timestamp'].max().date()}")
print(f"Test:            {len(df_test):,}  ({len(df_test)/n*100:.1f}%)  "
      f"{df_test['timestamp'].min().date()} → {df_test['timestamp'].max().date()}")

print(f"\nLabel distribution by split:")
for name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    counts = split_df['label'].value_counts().sort_index()
    pcts   = (counts / len(split_df) * 100).round(2)
    print(f"  {name}: normal={counts[0]:,} ({pcts[0]:.2f}%)  "
          f"attack={counts[1]:,} ({pcts[1]:.2f}%)  "
          f"fault={counts[2]:,} ({pcts[2]:.2f}%)")

Total rows:      1,323,608
Train:           794,164  (60.0%)  2020-07-07 → 2020-08-02
Validation:      264,722  (20.0%)  2020-08-02 → 2020-08-07
Test:            264,722  (20.0%)  2020-08-07 → 2020-08-10

Label distribution by split:
  Train: normal=729,244 (91.83%)  attack=8,947 (1.13%)  fault=55,973 (7.05%)


KeyError: 1

 ### THIS WAS STOPPED, I FOUND THAT THIS DATASET IS NOT INTENDED FOR WHAT I NEED IT FOR